In [ ]:
!pip install -q geemap rasterio

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import os
import shutil

from pathlib import Path
from google.colab import drive

# Mount Google Drive so the patches are not lost when Colab closes
drive.mount("/content/drive")

# Replace this with the Google Earth Engine project ID used in your labs
PROJECT_ID = "geog761-sep"

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print("Earth Engine connected successfully.")

In [ ]:
import requests
import pandas as pd
from pathlib import Path

# ArcGIS World Port Index point layer
query_url = (
    "https://services-eu1.arcgis.com/BuS9rtTsYEV5C0xh/"
    "arcgis/rest/services/World_Port_Index/FeatureServer/0/query"
)

all_records = []
offset = 0
page_size = 2000

while True:
    parameters = {
        "where": "1=1",
        "outFields": (
            "OBJECTID,INDEX_NO,PORT_NAME,COUNTRY,"
            "LATITUDE,LONGITUDE"
        ),
        "returnGeometry": "false",
        "resultOffset": offset,
        "resultRecordCount": page_size,
        "orderByFields": "OBJECTID",
        "f": "json"
    }

    response = requests.get(
        query_url,
        params=parameters,
        timeout=120
    )
    response.raise_for_status()

    result = response.json()

    if "error" in result:
        raise RuntimeError(result["error"])

    records = [
        feature["attributes"]
        for feature in result.get("features", [])
    ]

    all_records.extend(records)

    if len(records) < page_size:
        break

    offset += page_size

world_ports = pd.DataFrame(all_records)

print("All World Port Index ports:", len(world_ports))
display(world_ports.head())

In [ ]:
# Filter the World Port Index to Pacific Island ports and save as CSV

from pathlib import Path

pacific_country_values = {
    # Two-letter country codes
    "AS",  # American Samoa
    "CK",  # Cook Islands
    "FJ",  # Fiji
    "PF",  # French Polynesia
    "GU",  # Guam
    "KI",  # Kiribati
    "MH",  # Marshall Islands
    "FM",  # Federated States of Micronesia
    "NR",  # Nauru
    "NC",  # New Caledonia
    "NU",  # Niue
    "MP",  # Northern Mariana Islands
    "PW",  # Palau
    "PG",  # Papua New Guinea
    "WS",  # Samoa
    "SB",  # Solomon Islands
    "TK",  # Tokelau
    "TO",  # Tonga
    "TV",  # Tuvalu
    "VU",  # Vanuatu
    "WF",  # Wallis and Futuna

    # Full country names
    "AMERICAN SAMOA",
    "COOK ISLANDS",
    "FIJI",
    "FRENCH POLYNESIA",
    "GUAM",
    "KIRIBATI",
    "MARSHALL ISLANDS",
    "MICRONESIA, FEDERATED STATES OF",
    "FEDERATED STATES OF MICRONESIA",
    "NAURU",
    "NEW CALEDONIA",
    "NIUE",
    "NORTHERN MARIANA ISLANDS",
    "PALAU",
    "PAPUA NEW GUINEA",
    "SAMOA",
    "SOLOMON ISLANDS",
    "TOKELAU",
    "TONGA",
    "TUVALU",
    "VANUATU",
    "WALLIS AND FUTUNA"
}

# Standardise the country field
world_ports["COUNTRY_FILTER"] = (
    world_ports["COUNTRY"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Select Pacific Island ports
pacific_ports = (
    world_ports[
        world_ports["COUNTRY_FILTER"].isin(
            pacific_country_values
        )
    ]
    .drop(columns="COUNTRY_FILTER")
    .dropna(subset=["LATITUDE", "LONGITUDE"])
    .drop_duplicates(
        subset=["PORT_NAME", "LATITUDE", "LONGITUDE"]
    )
    .reset_index(drop=True)
)

# Create the Google Drive output folder
output_folder = Path(
    "/content/drive/MyDrive/GEOG761_port_training"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

# Save the complete and Pacific-only datasets
world_ports.drop(
    columns="COUNTRY_FILTER",
    errors="ignore"
).to_csv(
    output_folder / "world_port_index_all.csv",
    index=False
)

pacific_ports.to_csv(
    output_folder / "pacific_ports.csv",
    index=False
)

print("All World Port Index ports:", len(world_ports))
print("Pacific ports found:", len(pacific_ports))
print("Saved to:", output_folder / "pacific_ports.csv")

display(pacific_ports.head(20))

In [ ]:
# Define the selected port-disaster events.
# Port coordinates are taken directly from the World Port Index.

port_events = [
    {
        "sample_id": "port_vila_judy_kevin_2023",
        "wpi_port_name": "PORT VILA",
        "country_code": "VU",
        "disaster": "Tropical Cyclones Judy and Kevin",
        "pre_start": "2023-01-10",
        "pre_end": "2023-02-25",
        "post_start": "2023-03-06",
        "post_end": "2023-04-20",
        "label": -1
    },
    {
        "sample_id": "luganville_harold_2020",
        "wpi_port_name": "LUGANVILLE",
        "country_code": "VU",
        "disaster": "Tropical Cyclone Harold",
        "pre_start": "2020-02-20",
        "pre_end": "2020-04-05",
        "post_start": "2020-04-07",
        "post_end": "2020-05-20",
        "label": -1
    },
    {
        "sample_id": "honiara_harold_2020",
        "wpi_port_name": "HONIARA",
        "country_code": "SB",
        "disaster": "Tropical Cyclone Harold",
        "pre_start": "2020-02-15",
        "pre_end": "2020-03-31",
        "post_start": "2020-04-05",
        "post_end": "2020-05-20",
        "label": -1
    },
    {
        "sample_id": "apia_gita_2018",
        "wpi_port_name": "APIA",
        "country_code": "WS",
        "disaster": "Tropical Cyclone Gita",
        "pre_start": "2017-12-25",
        "pre_end": "2018-02-07",
        "post_start": "2018-02-12",
        "post_end": "2018-03-28",
        "label": -1
    },
    {
        "sample_id": "pago_pago_gita_2018",
        "wpi_port_name": "PAGO PAGO",
        "country_code": "AS",
        "disaster": "Tropical Cyclone Gita",
        "pre_start": "2017-12-25",
        "pre_end": "2018-02-07",
        "post_start": "2018-02-12",
        "post_end": "2018-03-28",
        "label": -1
    }
]

selected_rows = []

for event in port_events:

    port_names = (
        pacific_ports["PORT_NAME"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    country_codes = (
        pacific_ports["COUNTRY"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Use contains because some World Port Index names
    # may include words such as HARBOUR or HARBOR
    matching_port = pacific_ports[
        port_names.str.contains(
            event["wpi_port_name"],
            regex=False
        )
        &
        (country_codes == event["country_code"])
    ]

    if matching_port.empty:
        print(
            "Warning: port not found:",
            event["wpi_port_name"],
            event["country_code"]
        )
        continue

    port_record = matching_port.iloc[0]

    selected_rows.append({
        "sample_id": event["sample_id"],
        "port_name": port_record["PORT_NAME"],
        "country": port_record["COUNTRY"],
        "world_port_index": port_record["INDEX_NO"],
        "latitude": float(port_record["LATITUDE"]),
        "longitude": float(port_record["LONGITUDE"]),
        "disaster": event["disaster"],
        "pre_start": event["pre_start"],
        "pre_end": event["pre_end"],
        "post_start": event["post_start"],
        "post_end": event["post_end"],
        "label": event["label"]
    })

ports = pd.DataFrame(selected_rows)

print("Number of port-event rows:", len(ports))

display(ports)

In [ ]:
PATCH_SIZE_PIXELS = 128
PATCH_SIZE_METRES = 1280

# Start with the requested 50% threshold, but treat it as a parameter to test.
WATER_OCCURRENCE_THRESHOLD = 50
NODATA_VALUE = -9999.0

BAND_NAMES = [
    "pre_VV",
    "pre_VH",
    "post_VV",
    "post_VH",
    "change_VV",
    "change_VH"
]

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/GEOG761_port_training/Sentinel1_patches"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

surface_water_occurrence = (
    ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
    .select("occurrence")
)


def get_local_utm_crs(latitude, longitude):
    """Return the local WGS84 UTM CRS as an EPSG string."""

    zone = int((float(longitude) + 180) // 6) + 1
    zone = min(max(zone, 1), 60)
    epsg = (32600 if float(latitude) >= 0 else 32700) + zone
    return f"EPSG:{epsg}"


def make_patch_aoi(row):
    """Create a 1,280 m square in the port's local UTM projection."""

    latitude = float(row["latitude"])
    longitude = float(row["longitude"])
    export_crs = get_local_utm_crs(latitude, longitude)
    projection = ee.Projection(export_crs)

    point = ee.Geometry.Point([longitude, latitude])
    point_projected = point.transform(projection, 1)

    aoi = (
        point_projected
        .buffer(PATCH_SIZE_METRES / 2, 1)
        .bounds(1, projection)
    )

    return aoi, export_crs


def get_s1_collection(
    aoi,
    start_date,
    end_date,
    orbit_pass,
    relative_orbit=None
):
    """Obtain homogeneous 10 m Sentinel-1 IW images with VV and VH."""

    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("resolution_meters", 10))
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VV"
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VH"
            )
        )
        .filter(ee.Filter.eq("orbitProperties_pass", orbit_pass))
    )

    if relative_orbit is not None:
        collection = collection.filter(
            ee.Filter.eq(
                "relativeOrbitNumber_start",
                int(relative_orbit)
            )
        )

    return collection.select(["VV", "VH"])


def choose_orbit(aoi, pre_start, pre_end, post_start, post_end):
    """Select one pass and one relative orbit represented in both periods."""

    candidates = []

    for orbit_pass in ["ASCENDING", "DESCENDING"]:
        pre_collection = get_s1_collection(
            aoi, pre_start, pre_end, orbit_pass
        )
        post_collection = get_s1_collection(
            aoi, post_start, post_end, orbit_pass
        )

        pre_histogram = (
            pre_collection
            .aggregate_histogram("relativeOrbitNumber_start")
            .getInfo()
        )
        post_histogram = (
            post_collection
            .aggregate_histogram("relativeOrbitNumber_start")
            .getInfo()
        )

        common_orbits = set(pre_histogram) & set(post_histogram)

        for relative_orbit_key in common_orbits:
            pre_count = int(pre_histogram[relative_orbit_key])
            post_count = int(post_histogram[relative_orbit_key])

            candidates.append({
                "orbit": orbit_pass,
                "relative_orbit": int(float(relative_orbit_key)),
                "pre_count": pre_count,
                "post_count": post_count,
                "usable_count": min(pre_count, post_count),
                "total_count": pre_count + post_count
            })

    if not candidates:
        return None

    return max(
        candidates,
        key=lambda item: (
            item["usable_count"],
            item["total_count"]
        )
    )


def create_land_mask():
    """Return a fully valid binary mask: 1 land, 0 frequent water."""

    # Start from an unmasked constant. `where` changes it only where the
    # JRC band is valid and meets the threshold. This avoids propagating
    # the occurrence band's unusual fractional mask into the SAR bands.
    water_mask = (
        ee.Image.constant(0)
        .where(
            surface_water_occurrence.gte(
                WATER_OCCURRENCE_THRESHOLD
            ),
            1
        )
        .rename("frequent_water")
        .toByte()
    )

    return water_mask.Not().rename("land_mask")


land_mask = create_land_mask()


def create_six_band_patch(row):
    """Create masked pre, post and dB-change bands for one patch."""

    aoi, export_crs = make_patch_aoi(row)

    orbit_info = choose_orbit(
        aoi,
        row["pre_start"],
        row["pre_end"],
        row["post_start"],
        row["post_end"]
    )

    if orbit_info is None:
        return None, None, None

    orbit_pass = orbit_info["orbit"]
    relative_orbit = orbit_info["relative_orbit"]

    pre = (
        get_s1_collection(
            aoi,
            row["pre_start"],
            row["pre_end"],
            orbit_pass,
            relative_orbit
        )
        .median()
        .select(["VV", "VH"])
    )

    post = (
        get_s1_collection(
            aoi,
            row["post_start"],
            row["post_end"],
            orbit_pass,
            relative_orbit
        )
        .median()
        .select(["VV", "VH"])
    )

    # COPERNICUS/S1_GRD is in dB, so subtraction is a dB log-ratio.
    change = post.subtract(pre).rename([
        "change_VV",
        "change_VH"
    ])

    six_band_image = (
        pre.rename(["pre_VV", "pre_VH"])
        .addBands(post.rename(["post_VV", "post_VH"]))
        .addBands(change)
        .toFloat()
        .updateMask(land_mask)
        .unmask(NODATA_VALUE)
        .clip(aoi)
    )

    orbit_info = {
        **orbit_info,
        "export_crs": export_crs,
        "water_occurrence_threshold": WATER_OCCURRENCE_THRESHOLD,
        "nodata_value": NODATA_VALUE
    }

    return six_band_image, aoi, orbit_info


print("Functions and permanent-water mask created.")

In [ ]:
import requests
import shutil
from pathlib import Path

# Save locally in Colab first
LOCAL_OUTPUT_DIR = Path("/content/Sentinel1_patches")
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Google Drive backup location
DRIVE_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/GEOG761_port_training/Sentinel1_patches"
)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metadata_records = []

for _, row in ports.iterrows():
    print(f"Processing: {row['port_name']}")

    image, aoi, orbit_info = create_six_band_patch(row)

    if image is None:
        print("Skipped: no common relative orbit in both periods.")
        continue

    filename = f"{row['sample_id']}.tif"
    local_path = LOCAL_OUTPUT_DIR / filename
    drive_path = DRIVE_OUTPUT_DIR / filename

    download_url = image.getDownloadURL({
        "name": row["sample_id"],
        "region": aoi,
        "dimensions": [PATCH_SIZE_PIXELS, PATCH_SIZE_PIXELS],
        "crs": orbit_info["export_crs"],
        "format": "GEO_TIFF",
        "filePerBand": False
    })

    response = requests.get(download_url, timeout=300)
    response.raise_for_status()

    with open(local_path, "wb") as file:
        file.write(response.content)

    # Record NoData and band names in the GeoTIFF itself.
    with rasterio.open(local_path, "r+") as dataset:
        dataset.nodata = NODATA_VALUE
        for band_number, band_name in enumerate(BAND_NAMES, start=1):
            dataset.set_band_description(band_number, band_name)

    with rasterio.open(local_path) as dataset:
        if dataset.count != 6:
            raise ValueError(f"Expected 6 bands, found {dataset.count}")
        if (dataset.height, dataset.width) != (
            PATCH_SIZE_PIXELS,
            PATCH_SIZE_PIXELS
        ):
            raise ValueError(
                "Unexpected raster dimensions: "
                f"{dataset.height} x {dataset.width}"
            )

    shutil.copy2(local_path, drive_path)

    metadata_records.append({
        **row.to_dict(),
        "orbit_pass": orbit_info["orbit"],
        "relative_orbit": orbit_info["relative_orbit"],
        "number_pre_images": orbit_info["pre_count"],
        "number_post_images": orbit_info["post_count"],
        "water_occurrence_threshold": orbit_info[
            "water_occurrence_threshold"
        ],
        "nodata_value": orbit_info["nodata_value"],
        "export_crs": orbit_info["export_crs"],
        "image_file": filename
    })

    print("Saved locally:", local_path)
    print("Saved to Drive:", drive_path)

metadata = pd.DataFrame(metadata_records)

local_metadata_path = LOCAL_OUTPUT_DIR / "metadata.csv"
drive_metadata_path = DRIVE_OUTPUT_DIR / "metadata.csv"

metadata.to_csv(local_metadata_path, index=False)
metadata.to_csv(drive_metadata_path, index=False)

print("\nFiles in local folder:")
print(list(LOCAL_OUTPUT_DIR.glob("*")))

metadata

In [ ]:
# Display the five current Sentinel-1 port patches

titles = [
    "Pre-disaster VV",
    "Pre-disaster VH",
    "Post-disaster VV",
    "Post-disaster VH",
    "Change in VV",
    "Change in VH"
]

current_files = [
    LOCAL_OUTPUT_DIR / f"{sample_id}.tif"
    for sample_id in ports["sample_id"]
]

existing_files = [
    path for path in current_files
    if path.exists()
]

print("Current GeoTIFF files found:", len(existing_files))

if len(existing_files) == 0:
    print("No current GeoTIFF files were found.")

else:
    for example_path in existing_files:
        with rasterio.open(example_path) as src:
            patch = src.read().astype(np.float32)
            nodata_value = src.nodata

            print()
            print("Filename:", example_path.name)
            print("Shape:", patch.shape)
            print("Number of bands:", src.count)
            print("Coordinate system:", src.crs)
            print("Pixel size:", src.res)
            print("NoData value:", nodata_value)

        if nodata_value is not None:
            patch[patch == nodata_value] = np.nan
        else:
            patch[patch == NODATA_VALUE] = np.nan

        valid_land_fraction = np.isfinite(patch[0]).mean()
        print("Valid land fraction:", round(valid_land_fraction, 3))

        fig, axes = plt.subplots(2, 3, figsize=(13, 8))

        for band_number, ax in enumerate(axes.flat):
            if band_number < 4:
                plot = ax.imshow(
                    patch[band_number],
                    cmap="gray",
                    vmin=-25,
                    vmax=5
                )
            else:
                plot = ax.imshow(
                    patch[band_number],
                    cmap="RdBu_r",
                    vmin=-8,
                    vmax=8
                )

            ax.set_title(titles[band_number])
            ax.axis("off")
            plt.colorbar(plot, ax=ax, fraction=0.046, pad=0.04)

        port_title = example_path.stem.replace("_", " ").title()
        plt.suptitle(port_title, fontsize=15, fontweight="bold")
        plt.tight_layout()
        plt.show()

In [ ]:
# Create and download a 3 × 3 patch grid around each port

import requests
import shutil
import numpy as np
import pandas as pd
from pathlib import Path

# Each patch is 1,280 m wide.
# A 640 m step produces 50% overlap between neighbouring patches.
GRID_STEP_METRES = 640

GRID_OFFSETS = [
    -GRID_STEP_METRES,
    0,
    GRID_STEP_METRES
]

GRID_LOCAL_DIR = Path(
    "/content/Sentinel1_grid_patches"
)

GRID_DRIVE_DIR = Path(
    "/content/drive/MyDrive/"
    "GEOG761_port_training/Sentinel1_grid_patches"
)

GRID_LOCAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

GRID_DRIVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def offset_coordinates(
    latitude,
    longitude,
    north_offset_metres,
    east_offset_metres
):
    """
    Move a latitude/longitude coordinate by approximate
    north and east distances in metres.
    """

    metres_per_degree_latitude = 111320

    metres_per_degree_longitude = (
        111320
        * np.cos(np.radians(latitude))
    )

    new_latitude = (
        latitude
        + north_offset_metres
        / metres_per_degree_latitude
    )

    new_longitude = (
        longitude
        + east_offset_metres
        / metres_per_degree_longitude
    )

    return new_latitude, new_longitude


# Create the 45 candidate grid locations

grid_rows = []

for _, port in ports.iterrows():

    for row_number, north_offset in enumerate(
        GRID_OFFSETS,
        start=1
    ):

        for column_number, east_offset in enumerate(
            GRID_OFFSETS,
            start=1
        ):

            patch_latitude, patch_longitude = (
                offset_coordinates(
                    float(port["latitude"]),
                    float(port["longitude"]),
                    north_offset,
                    east_offset
                )
            )

            grid_rows.append({
                "sample_id": (
                    f"{port['sample_id']}"
                    f"_r{row_number}_c{column_number}"
                ),
                "source_port_id": port["sample_id"],
                "port_name": port["port_name"],
                "country": port["country"],
                "world_port_index": port["world_port_index"],
                "latitude": patch_latitude,
                "longitude": patch_longitude,
                "north_offset_m": north_offset,
                "east_offset_m": east_offset,
                "disaster": port["disaster"],
                "pre_start": port["pre_start"],
                "pre_end": port["pre_end"],
                "post_start": port["post_start"],
                "post_end": port["post_end"],

                # Remains unknown until visually inspected
                "label": -1
            })

grid_patches = pd.DataFrame(grid_rows)

print(
    "Candidate grid patches:",
    len(grid_patches)
)


# Download each six-band patch

download_records = []

for patch_number, (_, patch_row) in enumerate(
    grid_patches.iterrows(),
    start=1
):

    print(
        f"[{patch_number}/{len(grid_patches)}]",
        patch_row["sample_id"]
    )

    image, aoi, orbit_info = create_six_band_patch(
        patch_row
    )

    if image is None:
        print("  Skipped: no matching imagery.")
        continue

    filename = f"{patch_row['sample_id']}.tif"

    local_path = GRID_LOCAL_DIR / filename
    drive_path = GRID_DRIVE_DIR / filename

    try:
        download_url = image.getDownloadURL({
            "name": patch_row["sample_id"],
            "region": aoi,
            "dimensions": [
                PATCH_SIZE_PIXELS,
                PATCH_SIZE_PIXELS
            ],
            "crs": orbit_info["export_crs"],
            "format": "GEO_TIFF",
            "filePerBand": False
        })

        response = requests.get(
            download_url,
            timeout=300
        )

        response.raise_for_status()

        with open(local_path, "wb") as file:
            file.write(response.content)

        with rasterio.open(local_path, "r+") as dataset:
            dataset.nodata = NODATA_VALUE
            for band_number, band_name in enumerate(BAND_NAMES, start=1):
                dataset.set_band_description(band_number, band_name)

        shutil.copy2(
            local_path,
            drive_path
        )

        record = patch_row.to_dict()

        record.update({
            "orbit_pass": orbit_info["orbit"],
            "relative_orbit": orbit_info["relative_orbit"],
            "water_occurrence_threshold": WATER_OCCURRENCE_THRESHOLD,
            "nodata_value": NODATA_VALUE,
            "export_crs": orbit_info["export_crs"],
            "number_pre_images": orbit_info["pre_count"],
            "number_post_images": orbit_info["post_count"],
            "image_file": filename,
            "download_status": "success"
        })

        download_records.append(record)

        print("  Saved:", filename)

    except Exception as error:

        print("  Download failed:", error)


# Save grid metadata

grid_metadata = pd.DataFrame(download_records)

local_metadata_path = (
    GRID_LOCAL_DIR / "grid_metadata.csv"
)

drive_metadata_path = (
    GRID_DRIVE_DIR / "grid_metadata.csv"
)

grid_metadata.to_csv(
    local_metadata_path,
    index=False
)

grid_metadata.to_csv(
    drive_metadata_path,
    index=False
)

print()
print(
    "Successful grid patches:",
    len(grid_metadata)
)

print(
    "Saved to:",
    GRID_DRIVE_DIR
)

display(grid_metadata.head())